# GDELT backfill audit

Audit ingestion summaries for the historical GDELT backfill. Repeated observations are collapsed by `(stream, gdelt_timestamp)`, so retries and later skips do not count the same upstream file twice.

In [ ]:
from datetime import date
from pathlib import Path

import duckdb
import pandas as pd


MODE = "local" # gcp | local

LOG_ROOT = Path("../../data/logs/backfill/gdelt")
START_DATE = date(2022, 1, 1)
END_DATE = date(2026, 7, 7)
BYTES_PER_GB = 1_000_000_000
BYTES_PER_MB = 1_000_000
FILE_KEY = ["stream", "gdelt_timestamp"]

In [ ]:
file_summary_paths = sorted(
    LOG_ROOT.glob("run=*/ingestion/file_summary/*.parquet")
)
file_summary_glob = str(LOG_ROOT / "run=*/ingestion/file_summary/*.parquet")

file_summaries = duckdb.read_parquet(
    file_summary_glob,
    union_by_name=True,
).df()

file_summaries["gdelt_date"] = pd.to_datetime(
    file_summaries["gdelt_date"]
).dt.date

file_summaries = file_summaries[
    file_summaries["gdelt_date"].between(START_DATE, END_DATE)
]


pd.Series(
    {
        "file summary parquet files": len(file_summary_paths),
        "file rows in period": len(file_summaries),
    },
    name="count",
).to_frame()

In [ ]:
if MODE == "gcp":
    file_summaries["is_ready"] = file_summaries["gcp_url"].notna()
else: 
    file_summaries["is_ready"] = (
        file_summaries["download_status"].eq("skipped")
        | (
            file_summaries["download_status"].eq("downloaded")
            & file_summaries["unzip_status"].eq("extracted")
        )
    )    

In [ ]:
file_summaries["was_downloaded"] = file_summaries["download_status"].eq(
    "downloaded"
)
file_summaries["was_skipped"] = file_summaries["download_status"].eq("skipped")
file_summaries["had_404"] = file_summaries["http_status"].eq(404)

unique_files = (
    file_summaries.groupby(FILE_KEY, as_index=False)
    .agg(
        gdelt_date=("gdelt_date", "first"),
        url=("url", "first"),
        was_downloaded=("was_downloaded", "max"),
        was_skipped=("was_skipped", "max"),
        had_404=("had_404", "max"),
        zip_size_bytes=("zip_size_bytes", "max"),
        csv_size_bytes=("csv_size_bytes", "max"),
        observations=("url", "size"),
        is_ready=("is_ready", "max"),
    )
)

unique_files["unresolved_404"] = (
    unique_files["had_404"] & ~unique_files["is_ready"]
)
unique_files["zip_size_known"] = unique_files["zip_size_bytes"].fillna(0).gt(0)
unique_files["csv_size_known"] = unique_files["csv_size_bytes"].fillna(0).gt(0)

In [ ]:
files_expected = len(unique_files)
files_ready = unique_files["is_ready"].sum()

report = pd.Series(
    {
        "period start": START_DATE,
        "period end": END_DATE,
        "files expected": files_expected,
        "files downloaded": unique_files["was_downloaded"].sum(),
        "files in GCP": files_ready,
        "files ready": files_ready,
        "files not ready": files_expected - files_ready,
        "ready, %": round(files_ready / files_expected * 100, 4),
        "unique files with observed 404": unique_files["had_404"].sum(),
        "unique files still missing after retries": unique_files["unresolved_404"].sum(),
        "downloaded ZIP volume, GB": round(
            unique_files["zip_size_bytes"].fillna(0).sum() / BYTES_PER_GB,
            3,
        ),
        "extracted CSV volume with known size, GB": round(
            unique_files["csv_size_bytes"].fillna(0).sum() / BYTES_PER_GB,
            3,
        ),
        "ready files without ZIP size": (unique_files["is_ready"] & ~unique_files["zip_size_known"]).sum(),
        "ready files without CSV size": (unique_files["is_ready"] & ~unique_files["csv_size_known"]).sum(),
        "repeated file observations collapsed": len(file_summaries)
        - files_expected,
    },
    name="value",
).to_frame()

report

In [ ]:
summary_by_stream = (
    unique_files.groupby("stream")
    .agg(
        files_expected=("gdelt_timestamp", "size"),
        files_downloaded=("was_downloaded", "sum"),
        files_ready=("is_ready", "sum"),
        files_unresolved_404=("unresolved_404", "sum"),
        zip_size_bytes=("zip_size_bytes", "sum"),
        csv_size_bytes=("csv_size_bytes", "sum"),
    )
    .reset_index()
)

summary_by_stream["downloaded_zip_gb"] = (
    summary_by_stream.pop("zip_size_bytes") / BYTES_PER_GB
).round(3)
summary_by_stream["extracted_csv_gb"] = (
    summary_by_stream.pop("csv_size_bytes") / BYTES_PER_GB
).round(3)

summary_by_stream

In [ ]:
daily = (
    unique_files.groupby("gdelt_date", as_index=False)
    .agg(
        files_expected=("gdelt_timestamp", "size"),
        files_ready=("is_ready", "sum"),
        files_unresolved_404=("unresolved_404", "sum"),
    )
    .rename(columns={"gdelt_date": "date"})
)

calendar = pd.DataFrame(
    {"date": pd.date_range(START_DATE, END_DATE, freq="D").date}
)
daily = calendar.merge(daily, on="date", how="left")
count_columns = ["files_ready", "files_unresolved_404"]
daily[count_columns] = daily[count_columns].fillna(0).astype("int64")

fully_empty_dates = daily[
    daily["files_ready"] == 0
]
print(f"Fully empty dates: {len(fully_empty_dates)}")

fully_404_dates= daily[
    (daily["files_ready"] == 0)
    & (daily["files_unresolved_404"] == daily["files_expected"])
]
print(f"Fully empty dates with all files 404: {len(fully_404_dates)}")

display(fully_empty_dates)



In [ ]:
partial_dates = daily[(daily["files_ready"] > 0) & (daily["files_ready"] < 192)]
print(f"Partial dates: {len(partial_dates)}")
display(partial_dates)

In [ ]:
# Some files were skipped during reruns and not downloaded again, so their sizes are unknown.
known_zip_sizes = unique_files.loc[
    unique_files["is_ready"] & unique_files["zip_size_known"],
    "zip_size_bytes",
]
known_csv_sizes = unique_files.loc[
    unique_files["is_ready"] & unique_files["csv_size_known"],
    "csv_size_bytes",
]

median_zip_size = known_zip_sizes.median()
min_zip_size = known_zip_sizes.min()
max_zip_size = known_zip_sizes.max()

median_csv_size = known_csv_sizes.median()
min_csv_size = known_csv_sizes.min()
max_csv_size = known_csv_sizes.max()

print("Stat per file:\n")

print(
    f"Min zip size: {min_zip_size / BYTES_PER_MB:.2f} MB\n"
    f"Min csv size: {min_csv_size / BYTES_PER_MB:.2f} MB\n"
    f"Median zip size: {median_zip_size / BYTES_PER_MB:.2f} MB\n"
    f"Median csv size: {median_csv_size / BYTES_PER_MB:.2f} MB\n"
    f"Max zip size: {max_zip_size / BYTES_PER_MB:.2f} MB\n"
    f"Max csv size: {max_csv_size / BYTES_PER_MB:.2f} MB\n"
)

daily_size = (
    unique_files.groupby("gdelt_date", as_index=False)
    .agg(
        zip=("zip_size_bytes", "sum"),
        csv=("csv_size_bytes", "sum"),
        files_expected=("gdelt_timestamp", "size"),
        files_ready=("is_ready", "sum"),
        zip_sizes_known=("zip_size_known", "sum"),
        csv_sizes_known=("csv_size_known", "sum"),
    )
)

full_days = daily_size[
    (daily_size["files_ready"] == daily_size["files_expected"])
    & (daily_size["zip_sizes_known"] == daily_size["files_expected"])
    & (daily_size["csv_sizes_known"] == daily_size["files_expected"])
]

print("Stat per day:\n")
print(
    f"Min daily zip total: {full_days["zip"].min() / BYTES_PER_GB:.2f} GB\n"
    f"Min daily csv total: {full_days["csv"].min() / BYTES_PER_GB:.2f} GB\n"
    f"Median daily zip total: {full_days["zip"].median() / BYTES_PER_GB:.2f} GB\n"
    f"Median daily csv total: {full_days["csv"].median() / BYTES_PER_GB:.2f} GB\n"
    f"Max daily zip total: {full_days["zip"].max() / BYTES_PER_GB:.2f} GB\n"
    f"Max daily csv total: {full_days["csv"].max() / BYTES_PER_GB:.2f} GB\n"
)